## Mobility Robustness Optimization (MRO)

Takes in new observation data to train or update the bayesian digital twin models. It processes the input data and updates the model to better reflect the current network conditions.

Then MRO optimizes the mobility robustness by solving the underlying problem using the trained model: finding optimal `HYST` and `TTT`. There are two solve approaches shown: 

- Simple MRO
- Reinforced MRO

In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
import numpy as np
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO
from apps.mobility_robustness_optimization.mro_ml import BayesianMRO
from radp.digital_twin.utils.cell_selection import perform_attachment_hyst_ttt
from radp.digital_twin.utils.constants import RLF_THRESHOLD

*unzip the `data/mro_data.zip` file to get `data/mro_data/` folder*

# Showcasing **Simple MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [3]:
topology = pd.read_csv('data/mro_data/mro_topology.csv')
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [20]:
mobility_model_params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

<b>Optionally,</b> use mobility model to get `alpha` of your data and set it to params.

In [21]:
# [OPTIONAL] run this cell to get alpha calculated from the data into mobility_model_params

from radp.digital_twin.mobility.param_regression import get_predicted_alpha

# 20 UEs x 50 ticks = 1000 rows
ue_data = pd.read_csv("data/mro_data/UE_data_20UE_100ticks.csv") # change this to the data you want to use
ue_data = ue_data.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

# set random initial alpha
alpha0 = np.random.choice(np.arange(0, 1.1, 0.1))

alpha = get_predicted_alpha(ue_data, alpha0 = alpha0, seed = 42)

print(f"Learned alpha: {alpha:.2f}\n")

mobility_model_params["ue_tracks_generation"]["params"]["gauss_markov_params"]["alpha"] = alpha

Learned alpha: 0.40



/Users/tanzimfarhan/Desktop/Maveric/maveric/radp/digital_twin/mobility/param_regression.py:145: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  df = df.groupby("mock_ue_id").apply(calculate_distances_and_velocities)


In [22]:
mro = SimpleMRO(mobility_model_params, topology)

In [23]:
# initially bayesian_digital_twins is empty
print(f"bayesian_digital_twins: {mro.bayesian_digital_twins}")

bayesian_digital_twins: {}


- prepare `new_data` for training/updating `bayesian_digital_twins`

    - `new_data` should have received power data in cartesian df format. required cols ['latitude', 'longitude', 'cell_id', 'cell_rxpwr_dbm']

In [24]:
# 20 UEs x 100 ticks x 3 cells cartesian = 6000 rows
ue_data_with_rxpower = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_100ticks_train.csv") # change this to the data you want to use
input_data = ue_data_with_rxpower.copy()

input_data.head()

,longitude,latitude,cell_id,cell_rxpwr_dbm
0,-22.625309,59.806764,1,-100.311970
1,-22.625309,59.806764,2,-99.841523
2,-22.625309,59.806764,3,-99.432278
3,119.764151,54.857584,1,-100.294405
4,119.764151,54.857584,2,-100.132420


In [25]:
# train bayesian_digital_twins from scratch
mro.train_or_update_rf_twins(new_data=input_data)

[2025-07-14 13:21:55,548] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-07-14 13:21:55,583] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019079)
[2025-07-14 13:21:55,616] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019104)
[2025-07-14 13:21:55,649] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019147)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-07-14 13:21:55,683] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019230)
[2025-07-14 13:21:55,716] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019347)
[2025-07-14 13:21:55,747] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019529)
[2025-07-14 13:21:55,779] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019721)
[2025-07-14 13:21:55,815] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019941)
[2025-07-14 13:21:55,848] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020139)
[2025-07-14 13:21:55,881] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020338)
[2025-07-14 13:21:55,915] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020548)
[2025-07-14 13:21:55,948] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020730)
[2025-07-14 13:21:55,980] INFO:  Iter 14/100 - Loss: 0.526 (delta=-0.020908)
[2025-07-14 13:21:56,012] INFO:  Iter 15/100 - Loss: 0.505 (delta=-0.021091)
[2025-07-14 13:21:56,044] INFO:  Iter 16/100 - Loss: 0.483 (delta=-0.021269)
[2025-07-14 13:21:56,076] INFO:  Iter 17/100 - Loss: 0.462 (delta=-0.021430)
[202


Bayesian Digital Twins trained successfully.


In [26]:
mro.bayesian_digital_twins # bayesian_digital_twins is trained for each cell_id

{'cell_1': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x344e98110>,
 'cell_2': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1189cd7d0>,
 'cell_3': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x345ea0150>}

can save trained/updated `bayesian_digital_twins`

In [27]:
saving_dir_relative_path = "data/mro_data/"

mro.save_bdt(saving_dir_relative_path) # True indicates save is successful

Twins Saved Successfully as Pickle at: /Users/tanzimfarhan/Desktop/Maveric/maveric/notebooks/data/mro_data/digital_twins.pkl


True

call `solve()` method to get optimized `HYST` and `TTT`

In [ ]:
# adjust n_epochs for better performance
hyst, ttt = mro.solve(n_epochs=100)

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      6.6709380371   6      50.000000   
1      3.5088437366   50     49.150000   
2      3.2507041003   23     49.700000   
3      5.4379206735   30     49.550000   
4      1.1093751804   47     49.200000   
5      6.4638752881   22     49.750000   
6      4.3920004019   39     49.350000   
7      2.7623911406   35     49.450000   
8      6.7270838539   44     49.250000   
9      0.1461094661   6      50.000000   
10     3.2922500528   46     49.200000   
11     4.3289544170   14     49.950000   
12     0.0718687006   50     49.150000   
13     5.2502067474   44     49.250000   
14     2.2476419206   28     49.600000   
15     4.6450595301   48     49.200000   
16     4.6303521500   30     49.550000   
17     7.0401248727   8      50.000000   
18     6.0449542262   37     49.350000   
19     6.0148621258   34     49.500000   
20     1.3232986665   29     49.550000   
21     0.8677276208   17     49.85

In [ ]:
from notebooks.radp_library import mro_plot_scatter, plot_sinr_db_by_ue, mro_score_3d_plot
from radp.digital_twin.utils.constants import RLF_THRESHOLD
from radp.digital_twin.utils.cell_selection import perform_attachment_hyst_ttt

attached_df = perform_attachment_hyst_ttt(mro.simulation_data, hyst, ttt, RLF_THRESHOLD)
mro_plot_scatter(attached_df, topology)

In [ ]:
mro_score_3d_plot(mro.score)

In [ ]:
ue_id = 0 # change this to the UE you want to plot
plot_sinr_db_by_ue(attached_df, mro.simulation_data, ue_id)

can load this `bayesian_digital_twins` later when needed

In [ ]:
mro.bayesian_digital_twins = {} # bayesian_digital_twins is empty again

pkl_file_path = "data/mro_data/digital_twins.pkl"
mro.load_bdt(pkl_file_path) # True indicates load is successful

In [ ]:
# Dummy solve call to avoid fantasy observation error: ensuring all test independent caches exist
mro.solve(n_epochs=2)

let's try updating the `bayesian_digital_twins` with new observations

In [ ]:
# 20 UEs x 100 ticks x 3 cells cartesian = 6000 rows
new_obeservations = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_100ticks_update.csv") # change this to the data you want to use
input_data = new_obeservations.copy()

input_data.head()

In [ ]:
# update bdt with new observations, calling train_or_update_rf_twin() again
mro.train_or_update_rf_twins(input_data)

can solve with updated `bayesian_digital_twins`

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=100)

# Showcasing **Reinforced MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [ ]:
rl_mro = ReinforcedMRO(mobility_model_params, topology)

In [ ]:
print(f"bayesian_digital_twins: {rl_mro.bayesian_digital_twins}", end='\n\n') # bayesian_digital_twins is empty initially

pkl_file_path = "data/mro_data/digital_twins.pkl" # run previous section to have this file
rl_mro.load_bdt(pkl_file_path) # True indicates load is successful

In [ ]:
rl_mro.bayesian_digital_twins

- load `new_data` for training/updating `bayesian_digital_twins`

    - `new_data` should have received power data in cartesian df format. required cols ['latitude', 'longitude', 'cell_id', 'cell_rxpwr_dbm']

In [ ]:
# 20 UEs x 50 ticks x 3 cells cartesian = 3000 rows
ue_data_with_rxpower = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_50ticks.csv")
input_data = ue_data_with_rxpower.copy()

input_data

In [ ]:
# Dummy solve call to avoid fantasy observation error: ensuring all test independent caches exist
rl_mro.solve(total_timesteps=2)

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twins(input_data)

Solve using updated `bayesian_digital_twins`

In [ ]:
# adjust total_timesteps for better performance
hyst, ttt = rl_mro.solve(total_timesteps=100)

In [ ]:
attached_df = perform_attachment_hyst_ttt(mro.simulation_data, hyst, ttt, RLF_THRESHOLD)
mro_plot_scatter(attached_df, topology)

In [ ]:
ue_id = 0 # change this to the UE you want to plot
plot_sinr_db_by_ue(attached_df, mro.simulation_data, ue_id)

# Showcasing **Bayesian or XGBoost MRO** Solving Approach

use the following example `topology` and `mobility_model_params` to initiate MRO.

In [ ]:
# 20 UEs x 100 ticks x 3 cells cartesian = 6000 rows
ue_data_with_rxpower = pd.read_csv("data/mro_data/UE_data_with_rxpower_20UE_100ticks_train.csv") # change this to the data you want to use
input_data = ue_data_with_rxpower.copy()

input_data.head()

,longitude,latitude,cell_id,cell_rxpwr_dbm
0,-22.625309,59.806764,1,-100.311970
1,-22.625309,59.806764,2,-99.841523
2,-22.625309,59.806764,3,-99.432278
3,119.764151,54.857584,1,-100.294405
4,119.764151,54.857584,2,-100.132420


In [ ]:
n = len(input_data)
split_idx = int(n * 0.8)
train_data = input_data.iloc[:split_idx].reset_index(drop=True)
test_data = input_data.iloc[split_idx:].reset_index(drop=True)
print(f'Train shape: {train_data.shape}, Test shape: {test_data.shape}')


Train shape: (4800, 4), Test shape: (1200, 4)


In [ ]:
mro = BayesianMRO(mobility_model_params, topology)


In [ ]:
mro.train_or_update_rf_twins(train_data)

[2025-07-14 13:20:33,352] INFO:  Iter 1/100 - Loss: 0.773 (delta=inf)
[2025-07-14 13:20:33,378] INFO:  Iter 2/100 - Loss: 0.756 (delta=-0.017071)
[2025-07-14 13:20:33,399] INFO:  Iter 3/100 - Loss: 0.737 (delta=-0.018764)
[2025-07-14 13:20:33,415] INFO:  Iter 4/100 - Loss: 0.716 (delta=-0.021038)
[2025-07-14 13:20:33,430] INFO:  Iter 5/100 - Loss: 0.701 (delta=-0.015787)
[2025-07-14 13:20:33,446] INFO:  Iter 6/100 - Loss: 0.679 (delta=-0.022157)
[2025-07-14 13:20:33,462] INFO:  Iter 7/100 - Loss: 0.659 (delta=-0.019532)
[2025-07-14 13:20:33,478] INFO:  Iter 8/100 - Loss: 0.638 (delta=-0.021097)
[2025-07-14 13:20:33,492] INFO:  Iter 9/100 - Loss: 0.622 (delta=-0.016130)
[2025-07-14 13:20:33,508] INFO:  Iter 10/100 - Loss: 0.602 (delta=-0.019911)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-07-14 13:20:33,524] INFO:  Iter 11/100 - Loss: 0.579 (delta=-0.023314)
[2025-07-14 13:20:33,539] INFO:  Iter 12/100 - Loss: 0.560 (delta=-0.018574)
[2025-07-14 13:20:33,555] INFO:  Iter 13/100 - Loss: 0.539 (delta=-0.020572)
[2025-07-14 13:20:33,573] INFO:  Iter 14/100 - Loss: 0.518 (delta=-0.020893)
[2025-07-14 13:20:33,592] INFO:  Iter 15/100 - Loss: 0.496 (delta=-0.022288)
[2025-07-14 13:20:33,610] INFO:  Iter 16/100 - Loss: 0.476 (delta=-0.020052)
[2025-07-14 13:20:33,629] INFO:  Iter 17/100 - Loss: 0.456 (delta=-0.019647)
[2025-07-14 13:20:33,647] INFO:  Iter 18/100 - Loss: 0.437 (delta=-0.019634)
[2025-07-14 13:20:33,667] INFO:  Iter 19/100 - Loss: 0.410 (delta=-0.026431)
[2025-07-14 13:20:33,683] INFO:  Iter 20/100 - Loss: 0.391 (delta=-0.019909)
[2025-07-14 13:20:33,702] INFO:  Iter 21/100 - Loss: 0.369 (delta=-0.021330)
[2025-07-14 13:20:33,720] INFO:  Iter 22/100 - Loss: 0.347 (delta=-0.021950)
[2025-07-14 13:20:33,739] INFO:  Iter 23/100 - Loss: 0.320 (delta=-0.026904)


Bayesian Digital Twins trained successfully.


In [16]:
hyst, ttt = mro.solve()


Optimized Hyst: 2.928155259628293,
Optimized TTT: 6


In [15]:
mro.simulation_data = test_data

In [ ]:
attachment = perform_attachment_hyst_ttt(mro.simulation_data,hyst ,ttt, RLF_THRESHOLD)


In [ ]:
attachment

,ue_id,loc_x,loc_y,tick,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,distance_km,relative_bearing,cell_rxpower_dbm,sinr_db
32,0.0,48.510339,-16.462645,0.0,0.0,0.0,2.0,120.0,2100.0,15.543232,351.528854,-99.719653,-21.559878
65,1.0,19.286613,63.617111,0.0,90.0,180.0,3.0,240.0,2100.0,14.892870,100.713387,-99.366281,-21.304178
2,2.0,21.315702,-47.889952,0.0,-90.0,-180.0,1.0,0.0,2100.0,15.360440,201.315702,-99.623279,-21.503629
3,3.0,-70.559364,-79.511709,0.0,-90.0,-180.0,1.0,0.0,2100.0,13.970414,109.440636,-98.806769,-21.120596
4,4.0,-168.916011,-39.338640,0.0,-90.0,-180.0,1.0,0.0,2100.0,15.545318,11.083989,-99.694771,-21.400570
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,27.0,-14.856604,-5.730164,49.0,0.0,0.0,2.0,120.0,2100.0,14.386481,128.626615,-99.050519,-21.051832
28,28.0,-22.307315,40.806648,49.0,90.0,180.0,3.0,240.0,2100.0,15.515913,142.307315,-99.718214,-21.582111
29,29.0,-179.236707,41.434853,49.0,90.0,180.0,3.0,240.0,2100.0,15.503060,299.236707,-99.695899,-21.402736
30,30.0,89.100351,-11.651079,49.0,-90.0,-180.0,1.0,0.0,2100.0,15.981326,269.100351,-99.963924,-21.670178
